

# 3D Reconstruction

In this lab, we will experiment with 3D Structure-from-Motion systems, both feature-based and feed forward-based. More specifically, you will compare the performance of a state-of-the-art feature-based approach ([Colmap](https://colmap.github.io/index.html)) with a state-of-the-art feed forward-based approach ([Depth Anything 3 (DA3)](https://depth-anything-3.github.io/)).

## Speed and hardware
DA3 typically requires a significant amount of available GPU compute. Thus, in this exercise we only experiment with a small dataset, such that computation is feasible (within a few minutes) on the CPU.

Another option is to use [Google Colab](https://colab.research.google.com/). In the latter case, you will find comments with additional commands to run. Also, all the data (saved networks, datasets) will be auto-deleted after the end of the runtime session, so make sure to download the results of the training.

In [ ]:
# FOR COLAB: uncomment and run the following code every time you are starting the session
!pip install pycolmap
!pip install awesome-depth-anything-3

As a first step, let's download the data.

We are going to use the **Zurich Graffiti Dataset** available at https://github.com/tsattler/zurich_graffiti_dataset/ .


In [ ]:
# FOR COLAB: uncomment and run the following code every time you are starting the session
# for local: uncomment and run once
!wget https://github.com/tsattler/zurich_graffiti_dataset/raw/refs/heads/main/images.zip
!unzip images.zip

# Structure-from-Motion with Colmap

We will now build a 3D Structure-from-Motion model using Colmap. To this end, we will use pycolmap, a python interface to the Colmap library (the library itself is written in C++).  

We will need to perform the following steps:


*   Feature Extraction
*   Feature Matching & Spatial Verification
*   Incremental Structure-from-Motion

For each step, we will essentially call ready-made functionality from pycolmap (see [here](https://colmap.github.io/pycolmap/index.html#api) for pycolmap's API).

Pycolmap is storing all intermediate data (features, matches, etc.) in database. We will create this database automatically when extracting features.

In [ ]:
import pycolmap

# 1st step: Extract features and write them into a database.
# Use the extract_features function from pycolmap.
reader_options = pycolmap.ImageReaderOptions(camera_model="RADIAL")
camera_mode = "PER_IMAGE"
pycolmap.extract_features(#TODO)

# 2nd step: Exhaustive matching.
# Use the match_exhaustive function from pycolmap.
pycolmap.match_exhaustive(#TODO)

# 3rd step: Run incremental Structure-from-Motion.
# Use the incremental_mapping function from pycolmap. Note that Colmap
# potentially generates multiple 3D models. Thus, incremental_mapping returns
# a dictionary, where each entry consists of a model number and the
# corresponding reconstruction.
reconstructions = pycolmap.incremental_mapping(#TODO)

# Find the reconstruction that contains the largest number of images by
# looking at the number of registered images.
# TODO: Implement this functionality. The best model should be stored
# in a variable named best_recon

In [ ]:
# We will print the following statistics for the reconstruction:
# * The number of registered images.
# * The number of reconstructed 3D points.
# * The number of observations used to reconstruct the 3D points (the number of
#   features that were used to triangulate the 3D points).
# * The average track length for the points, i.e., the average number of
#   features used to triangulate the 3D point.
# * The average reprojection error of the 3D points.

def print_statistics(recon):
  # For printing the number of registered images and reconstructed 3D points, we
  # can directly use functions provided by the pycolmap.Reconstruction class.
  num_registered_images=#TODO: Get the number of registered images.
  num_3D_points=#TODO: Get the number of 3D points.
  print(f"Number of registered images: {num_registered_images}")
  print(f"Number of reconstructed 3D points: {num_3D_points}")

  # To obtain the number of observations and the average track length, we
  # iterate over all 3D points and look at the track length of each point.
  num_observations = 0
  # TODO: Compute the number of observations.
  for p_id, point3D in recon.points3D.items():

  print(f"Total number of observations: {num_observations}")
  print(f"Average track length: {num_observations/max(num_3D_points,1)}")

  # Similarly, for the average reprojection error, we iterate over all 3D points
  # and look at the error of each point.
  mean_reproj_error = 0.0
  # TODO: Compute the mean reprojection error.

  print(f"Mean reprojection error: {mean_reproj_error / float(max(num_3D_points,1))}")

# Print statistics for the Colmap model
print_statistics(best_recon)

You should get something close to 4,000 points triangulated from around 16,000 observations, with an average track length around 4. Of course, it is not bad if you get better numbers :)

Next, we are going to reconstruct the scene using DA3. You can find details on using DA3 [here](https://github.com/Aedelon/awesome-depth-anything-3) and the Python API [here](https://github.com/Aedelon/awesome-depth-anything-3/blob/main/docs/API.md).


In [ ]:
import torch, os, glob
from depth_anything_3.api import DepthAnything3
device = torch.device("cpu") # set to "gpu" if you have a powerful GPU.
# We will use the small model, which should be sufficient for such a small
# scene.
model = DepthAnything3.from_pretrained("depth-anything/DA3-SMALL")
model = model.to(device=device)
# TODO: Collect the paths to all images in a list called images.
# TODO: Run the network on the list of images to produce a set of predictions,
# store them in a variable named prediction .

Next, we want to compare the quality of the reconstruction obtained with DA3 with the reconstruction obtained via Colmap. Note that since both reconstructions can have arbirtary, and differing, scale factors, and since we do not have ground truth, we need to be a bit creative here.

We will compare the quality of the estimated poses indirectly using the metrics for Structure-from-Motion reconstructions implemented above. To be able to apply the function to the DA3 reconstruction, we will use its predicted poses and intrinsics to initialize an "empty" Colmap model that does not contain any 3D points. We will then use the database created by Colmap to triangulate 3D points using the poses and intrinsics predicted by DA3.  

In [ ]:
# Create a Colmap reconstruction for the camera poses predicted by DA3.
# For simplicity, we start off with the reconstruction generated by Colmap
# and modify the camera poses and intrinsics while removing all 3D points.
da3_recon = pycolmap.Reconstruction(reconstruction=best_recon)
# Notice that this will also clear best_recon.
da3_recon.delete_all_points2D_and_points3D()

# For each image: Add a new camera and image to the reconstruction.
for i in range(0, len(images)):
  # Finds the corresponding image ID in the da3_recon model.
  # Note that the image name should not contain the image
  # folder name, e.g., use name IMG_20151114_134548.jpg and not
  # name images/IMG_20151114_134548.jpg .
  image_id = da3_recon.find_image_with_name(images[i].split("/")[1]).image_id
  camera_id = da3_recon.images[image_id].camera_id
  frame_id = da3_recon.images[image_id].frame_id

  # Compute the Colmap pose for this image. DA3 provides estimated poses
  # under prediction.extrinsics. DA3 already provides the mapping from world
  # to camera coordinates required by Colmap. All we need to do is create a new
  # Rigid3d object that stores the pose estimated for the image.
  pose = # TODO: Obtain the pose

  # Update the camera pose by updating the frame with frame ID frame_id.
  da3_recon.frames[frame_id].set_cam_from_world(camera_id=camera_id,
                                                cam_from_world=pose)

  # Update the camera intrinsics by modifying the parameters of the camera
  # with camera ID camera_id.
  # The camera parameters should be stored as focal_length_x, focal_length_y,
  # principal_point_x, principal_point_y, radial. Set radial to 0.
  # DA3 provides the K matrix for the images under prediction.intrincis.
  # Notice that we need to rescale the intrinsics as DA3 downscales
  # the images before processing. The original images have size
  # 1024x768.
  scale_x = # TODO: Implement the scaling factor in x-direction (image width)
  scale_y = # TODO: Implement the scaling factor in y-direction (image height)
  K = # Obtain the intrinsic calibration matrix for this image.
  parameters = [# Rescale and store the parameters]
  # Set the parameters.
  da3_recon.cameras[camera_id].set_params_from_string(
      f"{parameters[0]}, {parameters[1]}, {parameters[2]}, {parameters[3]}")



In [ ]:
# Use the existing database and the newly created model to triangulate
# 3D points for the DA3 poses.
def retriangulate_reconstruction(recon):
  # Set parameters for the incremental reconstruction pipeline as to fix
  # intrinsics and extrinsics.
  incremental_pipeline_options = pycolmap.IncrementalPipelineOptions()
  incremental_pipeline_options.image_path = "images/"
  incremental_pipeline_options.load_all_images = True
  incremental_pipeline_options.fix_existing_frames = True
  incremental_pipeline_options.ba_refine_focal_length = False
  incremental_pipeline_options.ba_refine_principal_point = False
  incremental_pipeline_options.ba_refine_extra_params = False

  # pycolmap provides functionality to triangulate 3D points from a given
  # 3D reconstruction. We simply call it here. You will need to set an
  # output path where the reconstruction is stored.
  # TODO: Implement this functionality. Store the triangulated model in
  # a variable named triangulated_recon .

  return triangulated_recon

# Perform the actual retriangulation
triangulated_recon = retriangulate_reconstruction(da3_recon)

In [ ]:
# Finally, print statistics for the DA3 model.
print_statistics(triangulated_recon)

You should observe that the resulting model has fewer triangulated 3D points, a smaller number of observations, a smaller average track length, and a larger mean reprojection error than the Colmap model. This suggests that the poses estimated by DA3 are not as accurate as those predicted by Colmap.

Finally, we try to improve the reconstruction via bundle adjustment.

In [ ]:
# Bundle adjust the reconstruction we generated above. You can
# use readily-available pycolmap functionality.
# TODO: Implement the bundle adjustment.

# Print statistics for the reconstruction.
print_statistics(triangulated_recon)

You should observe that bundle adjustment is able to reduce the average reprojection error quite drastically, indicating that the refined poses are significantly better.

As a final step, we try to retriangulate the refined reconstruction again and print its statistics. What do you observe? Can you explain the behavior?

In [ ]:
retriangulated_recon2 = retriangulate_reconstruction(triangulated_recon)
print_statistics(retriangulated_recon2)

This completes this exercise. Please do not forget to submit your results.